<a href="https://www.kaggle.com/code/mrrogueknight/nutrition-health-survey-age-prediction?scriptVersionId=247006647" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [32]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nutrition-health-survey/Train_Data.csv
/kaggle/input/nutrition-health-survey/Test_Data.csv


In [33]:
# WARNING: This command will permanently delete all files and folders
# inside /kaggle/working/.

# First, list the files to see what will be deleted
print("--- Before Deletion ---")
!ls -l /kaggle/working/

# Now, delete everything inside the directory
!rm -rf /kaggle/working/*

# Verify that the directory is empty
print("\n--- After Deletion ---")
!ls -l /kaggle/working/

--- Before Deletion ---
total 4
-rw-r--r-- 1 root root 3121 Jun 23 17:55 submission.csv

--- After Deletion ---
total 0


In [34]:
import os

# List all files in the dataset directory
os.listdir("/kaggle/input/nutrition-health-survey")

['Train_Data.csv', 'Test_Data.csv']

In [35]:
train_df["age_group"].value_counts()

age_group
Adult     1638
Senior     314
Name: count, dtype: int64

In [36]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, make_scorer

# Load data
train_df = pd.read_csv("/kaggle/input/nutrition-health-survey/Train_Data.csv")
test_df = pd.read_csv("/kaggle/input/nutrition-health-survey/Test_Data.csv")

# Drop missing targets and encode labels
train_df = train_df.dropna(subset=["age_group"])
y = train_df["age_group"].map({'Adult': 0, 'Senior': 1})
X = train_df.drop(columns=["SEQN", "age_group"])
X_test = test_df.drop(columns=["SEQN"])
test_ids = test_df["SEQN"]

# Imputation
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X_test = imputer.transform(X_test)

# Scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# Class imbalance
scale = (y == 0).sum() / (y == 1).sum()

# XGBoost (slightly tuned but safe)
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=4,
    scale_pos_weight=scale,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# Optional: CV score check
f1 = make_scorer(f1_score)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, scoring=f1, cv=cv)
print(f"F1 CV Mean: {cv_scores.mean():.4f}")

# Train and predict
model.fit(X, y)
preds = model.predict(X_test)

# Submission
submission = pd.DataFrame({
    "SEQN": test_ids,
    "age_group": preds.astype(int)
})
submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.head()

F1 CV Mean: 0.3763


,SEQN,age_group
0,77017.0,0
1,75580.0,1
2,73820.0,0
3,80489.0,0
4,82047.0,0
